# Metadata for ablation

1. Use old code from `scripts/one_beat.py` to find heartrate for each subject.
2. Keep the filters from the old code
3. Explore heart rates of the subjects
4. Run `one_beat.py` on all the files from Zenodo before running this notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Run one_beat.py to get the summary dataframes - use all the files from Zenodo
# Read all the dataframes and combine them into one summary dataframe

hr_dataframes = []
for i in range(1, 17):
    df_summary = pd.read_csv(f'../data/peak_summary_{i}.csv')
    df_hr = df_summary.groupby('subject')[['hr', 'exam_id', 'retain_subject']].first().reset_index()
    df_hr.loc[:, 'file_num'] = i
    hr_dataframes.append(df_hr)
df_hr = pd.concat(hr_dataframes, ignore_index=True)
df_hr.head()


In [ ]:
df_hr = df_hr[df_hr['retain_subject'] == True]

In [ ]:
# make a histogram of heart rates
# remove all the subject with heart rate above 150 bpm
hr_filter = (df_hr['hr'] <= 200)
plt.figure(figsize=(10, 6))
plt.hist(df_hr[hr_filter]['hr'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Heart Rates')
plt.xlabel('Heart Rate (bpm)')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
df_hr.shape

In [ ]:
# 10 seconds are represented in 4096 pixels
# So we only have 3 significant digits in the the heart rate values


In [ ]:
# find how many non-unique values of hr are there
digits = 3
df_hr['hr'].round(digits).value_counts()[df_hr['hr'].round(digits).value_counts() > 1]

In [ ]:
df_meta = pd.read_csv('../data/exams.csv')
df_meta.head()

In [ ]:
df_hr = df_hr.merge(
    df_meta[['exam_id', 'age', 'is_male', 'normal_ecg', 'nn_predicted_age']],
    on='exam_id',
    how='inner'
)

In [ ]:
df_hr.head()

In [ ]:
data_filter = df_hr['normal_ecg']
df_hr[data_filter]['hr'].hist()
plt.show()

In [ ]:
df_hr[data_filter]['hr'].round(digits).value_counts()[df_hr[data_filter]['hr'].round(digits).value_counts() > 1]

In [ ]:
df_hr['hr_round'] = df_hr['hr'].round(3)
df_hr[
    (abs(df_hr['hr_round'] - 77.83) < 0.01)
]

In [ ]:
# assign identifiers to the subjects with the same rounded heart rate values
df_hr['hr_id'] = df_hr.groupby('hr_round')['subject'].transform('first')
df_hr.head()